# 16.6 Batch 放不下怎麼辦？


想讓三筆資料一起決定一次更新，記憶體卻只能同時放其中一兩筆，可以分次計算、最後再更新嗎？可以，但要把各份資料的誤差按原本的比例相加，不能平均「兩次各自的平均」。這種做法叫gradient accumulation（梯度累積）：多次向後計算都加到同一份梯度，最後只做一次參數更新。前置是[梯度暖身](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)；梯度表示改動權重對誤差的影響，PyTorch的backward預設會累加它。

用一個權重w=1與三個輸入 `[1,2,3]`，希望輸出全為零，誤差取各輸出的平方。整批平均誤差是 `(1+4+9)/3=14/3`，對w的梯度是 `2×(1+4+9)/3=28/3`，約9.3333。若拆成一筆與兩筆，第一份誤差和1，第二份誤差和13；兩份都除以共同總數3，累加仍是原來的14/3。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
whole = torch.tensor(1.0, requires_grad=True)
(whole * x).square().mean().backward()
accumulated = torch.tensor(1.0, requires_grad=True)
wrong = torch.tensor(1.0, requires_grad=True)
for part in (x[:1], x[1:]):
    ((accumulated * part).square().sum() / len(x)).backward()
    ((wrong * part).square().mean() / 2).backward()
print("整批梯度", round(whole.grad.item(), 4))
print("正確累積", round(accumulated.grad.item(), 4))
print("錯誤平均", round(wrong.grad.item(), 4))

`x[:1]`取索引1之前，也就是第一筆；`x[1:]`取索引1到結尾，也就是後兩筆，切片終點本身不包含在內。三個權重初值相同但獨立，分別核對三種計算。正確那行先取每份的誤差和，再除以共同的3；每次backward會把新貢獻加到accumulated.grad。錯誤那行先對份內求平均，再讓兩份各占一半，相當於第一筆占一半、第二與第三筆各占四分之一，已不是三筆等權。輸出應為9.3333、9.3333與7.5：程式能跑不代表平均方式正確。

真實語言模型通常按有效答案token計算誤差，這裡「有效」指[確實納入答案誤差的位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)，[PAD](https://birdhackor.github.io/tiny-perceptron-vlm/7.7.html)則是補齊長度的符號，不是要學的答案。有的micro-batch（拆出的較小批次）只有一個有效token，有的有十個，因此分母應是累積範圍內的有效token總數，不是迴圈次數，也不是包含PAD的表格大小。只有每份有效數完全相同時，「各份平均再平均」才會與整體平均相同。

訓練操作順序也很重要：先清除前一輪梯度，再對各份backward，最後才裁剪梯度（整份梯度過大時一起縮小，避免更新突然太大），再做一次step更新權重。若每份都step，後一份看到的是更新後的權重，就不再是同一個大batch。此例沒有step，只驗證梯度。有些運算依同一批資料共同計算平均值等統計，拆批就可能改變它；dropout則是在訓練時隨機關閉部分特徵，分次計算可能抽到不同遮罩。這些行為需另核對，不能把任何模型分批都視為無條件等價。

練習只改拆法為兩筆與一筆：`(x[:2],x[2:])`。先預測正確梯度仍9.3333，錯誤方式改為11.5，再執行核對。拆法不該影響同一整批的權重，如果結果變了，先檢查每筆資料到底被分配了多少份量。
